In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install langchain-chroma

In [ ]:
!pip install -q ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35 rapidfuzz python-dotenv sentence-transformers pandas langchain-text-splitters
print("[✓] Tüm kütüphaneler başarıyla kuruldu.")

In [ ]:
import os
import sys
import datetime
import time
import asyncio
import hashlib
import pandas as pd

from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import BaseMessage, AIMessage

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.run_config import RunConfig
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer, 
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

import shutil
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

In [ ]:
!cp -r /kaggle/input/notebooks/""/ragas-sonras-embedding/chroma_db .

In [ ]:
!cp -r /kaggle/input/notebooks/""/ragas-sonras-embedding/chroma_db_contextual .

In [ ]:
!cp /kaggle/input/notebooks/""/ragas-sonras-embedding/chunk_map_20260829_060905.csv .
!cp /kaggle/input/notebooks/""/ragas-sonras-embedding/testset_mixed_102_soru_20260829_060905.csv .

!mkdir outputs

In [ ]:
!pip install rank_bm25
!pip install sentence-transformers

In [ ]:
import ast
import time
import asyncio
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from rapidfuzz import fuzz, process
from openai import OpenAI
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.prompts import PromptTemplate

print("="*70)
print("   9'LU TAM KADRO RAG ANALİZİ (RERANKER VE MQ DAHİL)")
print("="*70)

# 1. Özel Azure Responses Model Tanımlaması
class CustomAzureResponsesModel(BaseChatModel):
    api_key: str = ""
    base_url: str = ""
    model_name: str = "gpt-4.1-mini"
    delay: float = 0.5 

    def _generate(self, messages: list[BaseMessage], stop: list[str] | None = None, **kwargs) -> ChatResult:
        time.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])
        
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)
        
        output_text = getattr(response, "output_text", str(response))
        message = AIMessage(content=output_text)
        return ChatResult(generations=[ChatGeneration(message=message)])

    async def _agenerate(self, messages: list[BaseMessage], stop: list[str] | None = None, **kwargs) -> ChatResult:
        await asyncio.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])
        
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)
        
        output_text = getattr(response, "output_text", str(response))
        message = AIMessage(content=output_text)
        return ChatResult(generations=[ChatGeneration(message=message)])

    @property
    def _llm_type(self) -> str:
        return "custom-azure-responses-openai"

# 2. Dosya Yolları ve Veri Hazırlığı
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"

df_map = pd.read_csv(chunk_map_path)
df_test = pd.read_csv(testset_path)

text_to_id = {}
docs_for_bm25 = []

for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    text_to_id[clean_text] = chunk_id
    docs_for_bm25.append(Document(
        page_content=clean_text,
        metadata={"chunk_id": chunk_id, "filename": str(row['filename'])}
    ))

doc_texts_list = list(text_to_id.keys())

FUZZY_THRESHOLD = 90
def match_text_to_id(gt_text, threshold=FUZZY_THRESHOLD):
    clean_gt = str(gt_text).strip()
    if clean_gt in text_to_id:
        return text_to_id[clean_gt], "exact", 100.0
    match = process.extractOne(clean_gt, doc_texts_list, scorer=fuzz.ratio)
    if match is not None:
        best_match, score, _ = match
        if score >= threshold:
            return text_to_id[best_match], "fuzzy", score
    return None, "no_match", 0.0

# 3. Veritabanı ve Model Bağlantıları
db_dir = "/kaggle/working/chroma_db"
contextual_db_dir = "/kaggle/working/chroma_db_contextual"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

db = Chroma(persist_directory=db_dir, embedding_function=embeddings)
db_contextual = Chroma(persist_directory=contextual_db_dir, embedding_function=embeddings)

contextual_data = db_contextual.get()
docs_for_ctx_bm25 = []
for doc_text, metadata in zip(contextual_data['documents'], contextual_data['metadatas']):
    docs_for_ctx_bm25.append(Document(
        page_content=doc_text,
        metadata={"chunk_id": str(metadata.get('chunk_id'))}
    ))

K = 5
INITIAL_K_FOR_RERANK = 15

# Retriever Tanımları
basic_retriever = db.as_retriever(search_kwargs={"k": K})

bm25_std = BM25Retriever.from_documents(docs_for_bm25)
bm25_std.k = K
chroma_std = db.as_retriever(search_kwargs={"k": K})
hybrid_std_retriever = EnsembleRetriever(
    retrievers=[bm25_std, chroma_std], weights=[0.5, 0.5]
)

contextual_dense_retriever = db_contextual.as_retriever(search_kwargs={"k": K})

bm25_ctx = BM25Retriever.from_documents(docs_for_ctx_bm25)
bm25_ctx.k = K
contextual_chroma_std = db_contextual.as_retriever(search_kwargs={"k": K})
contextual_hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_ctx, contextual_chroma_std], weights=[0.5, 0.5]
)

bm25_wide = BM25Retriever.from_documents(docs_for_bm25)
bm25_wide.k = INITIAL_K_FOR_RERANK
chroma_wide = db.as_retriever(search_kwargs={"k": INITIAL_K_FOR_RERANK})
hybrid_wide = EnsembleRetriever(
    retrievers=[bm25_wide, chroma_wide], weights=[0.5, 0.5]
)

bm25_ctx_wide = BM25Retriever.from_documents(docs_for_ctx_bm25)
bm25_ctx_wide.k = INITIAL_K_FOR_RERANK
contextual_wide_dense = db_contextual.as_retriever(search_kwargs={"k": INITIAL_K_FOR_RERANK})
contextual_hybrid_wide = EnsembleRetriever(
    retrievers=[bm25_ctx_wide, contextual_wide_dense], weights=[0.5, 0.5]
)

print("[*] Reranker modelleri yükleniyor...")
reranker_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={'device': 'cuda'}
)
compressor = CrossEncoderReranker(model=reranker_model, top_n=K)

reranker_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=hybrid_wide
)

# Contextual + Hibrit + Reranker (Saf / Wide tabanlı)
contextual_reranker_pure = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=contextual_hybrid_wide
)

print("[*] Özel Multi-Query LLM zinciri hazırlanıyor...")
llm = CustomAzureResponsesModel()

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate 3 
alternative versions of the given user query to retrieve relevant documents from a vector database. 
By generating multiple perspectives on the user query, your goal is to help the user overcome 
some of the limitations of distance-based similarity search. Provide these alternative questions separated by newlines.
Original question: {question}
Alternative questions:"""
)

def get_multi_query_docs(query, base_retriever):
    formatted_prompt = QUERY_PROMPT.format(question=query)
    response_msg = llm.invoke(formatted_prompt)
    response_text = response_msg.content
    
    queries = [q.strip() for q in response_text.strip().split("\n") if q.strip()]
    queries.append(query)
    
    all_docs = []
    seen_ids = set()
    for q in queries:
        docs = base_retriever.invoke(q)
        for doc in docs:
            doc_id = str(doc.metadata.get('chunk_id'))
            if doc_id not in seen_ids:
                seen_ids.add(doc_id)
                all_docs.append(doc)
    return all_docs

# 4. Test Döngüsü
results_data = []
skip_count = 0
total_soru = len(df_test)

print(f"\n[*] Toplam {total_soru} soru 9 farklı sistem üzerinde test ediliyor...\n" + "-"*60)

for index, row in df_test.iterrows():
    query = row['user_input']
    synth_name = row.get('synthesizer_name', row.get('question_type', 'Genel_Soru'))

    try:
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        skip_count += 1
        continue

    expected_ids = []
    for gt_text in ground_truth_texts:
        matched_id, _, _ = match_text_to_id(gt_text)
        if matched_id:
            expected_ids.append(matched_id)

    if len(expected_ids) == 0:
        skip_count += 1
        continue

    print(f"👉 Soru [{index+1}/{total_soru}] işleniyor...")

    def evaluate_retriever(retriever, system_name):
        print(f"   -> Test ediliyor: {system_name}")
        res = retriever.invoke(query)
        retrieved_ids = [str(doc.metadata.get('chunk_id')) for doc in res[:K]]
        true_positives = sum(1 for exp_id in expected_ids if exp_id in retrieved_ids)
        recall = true_positives / len(expected_ids) if len(expected_ids) > 0 else 0.0
        precision = true_positives / K if K > 0 else 0.0
        return recall, precision

    def evaluate_multiquery_retriever(base_retriever, system_name, is_rerank=False):
        print(f"   -> Multi-Query çalıştırılıyor: {system_name}")
        all_retrieved_docs = get_multi_query_docs(query, base_retriever)
        if is_rerank:
            compressed_docs = compressor.compress_documents(all_retrieved_docs, query)
            res = compressed_docs[:K]
        else:
            res = all_retrieved_docs[:K]
            
        retrieved_ids = [str(doc.metadata.get('chunk_id')) for doc in res[:K]]
        true_positives = sum(1 for exp_id in expected_ids if exp_id in retrieved_ids)
        recall = true_positives / len(expected_ids) if len(expected_ids) > 0 else 0.0
        precision = true_positives / K if K > 0 else 0.0
        return recall, precision

    # 1. Temel Vektör
    r_basic, p_basic = evaluate_retriever(basic_retriever, "1. Temel Vektör (Dense)")
    # 2. Normal Hibrit
    r_hybrid, p_hybrid = evaluate_retriever(hybrid_std_retriever, "2. Normal Hibrit (BM25+Vec)")
    # 3. Contextual Dense
    r_ctx_dense, p_ctx_dense = evaluate_retriever(contextual_dense_retriever, "3. Contextual RAG (Dense)")
    # 4. Contextual + Hibrit (Saf)
    r_ctx_hybrid_pure, p_ctx_hybrid_pure = evaluate_retriever(contextual_hybrid_retriever, "4. Contextual + Hibrit (Saf)")
    # 5. Contextual + Hibrit (MQ)
    r_ctx_hybrid_mq, p_ctx_hybrid_mq = evaluate_multiquery_retriever(contextual_hybrid_retriever, "5. Contextual + Hibrit (MQ)", is_rerank=False)
    # 6. Hibrit + Reranker
    r_rerank, p_rerank = evaluate_retriever(reranker_retriever, "6. Hibrit + Reranker")
    # 7. Contextual + Hibrit + Reranker (Saf)
    r_ctx_rerank_pure, p_ctx_rerank_pure = evaluate_retriever(contextual_reranker_pure, "7. Contextual + Hibrit + Reranker (Saf)")
    # 8. Contextual + Hibrit + Reranker + Multi-Query (MQ)
    r_ctx_rerank_mq, p_ctx_rerank_mq = evaluate_multiquery_retriever(contextual_hybrid_wide, "8. Ctx + Hibrit + Reranker + MQ", is_rerank=True)

    results_data.append({
        'query': query,
        'synthesizer_name': synth_name,
        'recall_basic': r_basic, 'precision_basic': p_basic,
        'recall_hybrid': r_hybrid, 'precision_hybrid': p_hybrid,
        'recall_ctx_dense': r_ctx_dense, 'precision_ctx_dense': p_ctx_dense,
        'recall_ctx_hybrid_pure': r_ctx_hybrid_pure, 'precision_ctx_hybrid_pure': p_ctx_hybrid_pure,
        'recall_ctx_hybrid_mq': r_ctx_hybrid_mq, 'precision_ctx_hybrid_mq': p_ctx_hybrid_mq,
        'recall_rerank': r_rerank, 'precision_rerank': p_rerank,
        'recall_ctx_rerank_pure': r_ctx_rerank_pure, 'precision_ctx_rerank_pure': p_ctx_rerank_pure,
        'recall_ctx_rerank_mq': r_ctx_rerank_mq, 'precision_ctx_rerank_mq': p_ctx_rerank_mq
    })
    print("-" * 50)

df_res = pd.DataFrame(results_data)

# 5. Genel Ortalama Karnesi
print("\n" + "="*75)
print("🎯 8'Lİ / 9'LU TAM KADRO SİSTEM ORTALAMA RECALL@5 & ADİL PRECISION@5 KARNESİ")
print("="*75)
print(f"1. Temel Vektör (Dense)              | Recall: %{df_res['recall_basic'].mean()*100:.1f} | Precision: %{df_res['precision_basic'].mean()*100:.1f}")
print(f"2. Normal Hibrit (BM25+Vec)          | Recall: %{df_res['recall_hybrid'].mean()*100:.1f} | Precision: %{df_res['precision_hybrid'].mean()*100:.1f}")
print(f"3. Contextual RAG (Dense)            | Recall: %{df_res['recall_ctx_dense'].mean()*100:.1f} | Precision: %{df_res['precision_ctx_dense'].mean()*100:.1f}")
print(f"4. Contextual + Hibrit (Saf)         | Recall: %{df_res['recall_ctx_hybrid_pure'].mean()*100:.1f} | Precision: %{df_res['precision_ctx_hybrid_pure'].mean()*100:.1f}")
print(f"5. Contextual + Hibrit (MQ)          | Recall: %{df_res['recall_ctx_hybrid_mq'].mean()*100:.1f} | Precision: %{df_res['precision_ctx_hybrid_mq'].mean()*100:.1f}")
print(f"6. Hibrit + Reranker                 | Recall: %{df_res['recall_rerank'].mean()*100:.1f} | Precision: %{df_res['precision_rerank'].mean()*100:.1f}")
print(f"7. Ctx + Hibrit + Reranker (Saf)     | Recall: %{df_res['recall_ctx_rerank_pure'].mean()*100:.1f} | Precision: %{df_res['precision_ctx_rerank_pure'].mean()*100:.1f}")
print(f"8. Ctx + Hibrit + Reranker + MQ      | Recall: %{df_res['recall_ctx_rerank_mq'].mean()*100:.1f} | Precision: %{df_res['precision_ctx_rerank_mq'].mean()*100:.1f}")
print("-" * 75)

df_res.to_csv("/kaggle/working/dev_kapsamli_rag_analiz_9lu_tam.csv", index=False, encoding="utf-8")
print("\n[✓] Tam kadro sonuçlar başarıyla kaydedildi.")

In [ ]:
!zip -r  /kaggle/working/chroma_db.zip /kaggle/working/chroma_db

In [ ]:
!zip -r /kaggle/working/chroma_db_contextual.zip /kaggle/working/chroma_db_contextual